In [ ]:
import requests as req
import pandas as pd

In [ ]:
#se configura header y pagina a scrappear#

headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36",
    "x-nextjs-data": "1"
}

response = req.get("https://www.fotmob.com/api/data/leagueseasondeepstats?id=38&season=27185&type=players&stat=goal_assist", 
             headers=headers)

print(response.status_code)



In [ ]:
#obtenemos los datos en formato json y los imprimimos para ver su estructura#

data = response.json()
print(data.keys())
print(type(data))

In [ ]:
print(type(data['statsData']))
print(data['statsData'][0])

In [ ]:
print(type(data['statsList']))
# Si es una lista, mira el primer elemento:
print(data['statsList'][13]) 
# O si es un diccionario, mira sus llaves:
# print(data['statsList'].keys())

In [ ]:
#sacamos los datos de id, nombre y valor de cada jugador y los guardamos en un dataframe#

asistencias = []

for jugador in (data['statsData']):
    id = jugador.get('id', {})
    nombre = jugador.get('name', {})
    valor = jugador.get('substatValue', {}).get('value')

    df_fila = {'id': id, 'nombre': nombre, 'valor': valor}

    asistencias.append(df_fila)

df_asistencias = pd.DataFrame(asistencias)
print(df_asistencias)

In [ ]:
stat_name = ["goals", "goal_assist", "expected_goals", "expected_goalsontarget", "total_scoring_att", "expected_assists_per_90","won_contest"
             , "accurate_long_balls", "poss_won_att_3rd", "defensive_contributions", "total_tackle", "interception",  "effective_clearance"]
league_id = [38,122]
season_id = [38611, 38000]
slug = ["bundesliga-players", "1-liga-players"]
build_id = "ZL64iOcIu6K0j4uv0G7J1"

endpoint = "https://www.fotmob.com/api/data/leagueseasondeepstats"

#f"https://www.fotmob.com/api/data/{build_id}/es/leagues/{league_id}/stats/season/{season_id}/players/{stat_name}/{slug}.json"

query_params = {
    "id": league_id,
    "season": season_id,
    "type": "players",
    "stat": stat_name,
    "slug": slug,
    "lng": "es"
}

datos_jugadores = []
for x in league_id:
    for i in stat_name:
        query_params["stat"] = i

        response = req.get(endpoint, headers=headers, params=query_params)
        data = response.json()


        for jugador in data['statsData']:
            id = jugador.get('id', {})
            nombre = jugador.get('name', {})
            valor = jugador.get('statValue', {}).get('value')

            df_fila = {'id': id, 'nombre': nombre, 'valor': valor, 'stat': i, "league": x}

            datos_jugadores.append(df_fila)

    df_datos_jugadores = pd.DataFrame(datos_jugadores)

    


In [ ]:
print(df_datos_jugadores[df_datos_jugadores['nombre'] == "Wiktor Nowak"])

In [ ]:
df_datos_jugadores_pivoted = df_datos_jugadores.pivot(index=['id', 'nombre'], columns='stat', values='valor').reset_index()
df_datos_jugadores_pivoted.isnull().sum()